# Airport Operations Dataset - Complete Data Audit
Loading and preparing the dataset for passenger flow prediction model

In [7]:
import pandas as pd
import numpy as np
from datetime import datetime

# Define column names for flights (CSV has numeric indices as header)
flights_columns = [
    'flight_id', 'airline_name', 'airline_code', 'origin', 'destination',
    'scheduled_departure', 'scheduled_arrival', 'actual_departure', 'actual_arrival',
    'aircraft_type', 'aircraft_registration', 'scheduled_capacity', 'actual_capacity',
    'flight_status', 'delay_minutes', 'delay_reason', 'terminal', 'gate',
    'on_tarmac', 'passengers_boarded', 'total_seats', 'gate_assignment_time',
    'crew_ready', 'operational_status', 'on_time_percentage', 'turnaround_time_min',
    'fuel_efficiency_ratio', 'time_of_day', 'day_of_week', 'holiday_flag',
    'season', 'flight_type'
]


#Load all datasets

print("\n Loading datasets...")

# Flights: Skip numeric header row and apply column names
flights = pd.read_csv(r"D:\Passengers flow predictor\Passenger_Flow_Predictor\data\flights.csv", header=None, names=flights_columns, skiprows=1)

# Other datasets: Load normally
passengers = pd.read_csv(r"D:\Passengers flow predictor\Passenger_Flow_Predictor\data\passengers.csv")
security = pd.read_csv(r"D:\Passengers flow predictor\Passenger_Flow_Predictor\data\security_screening.csv")
baggage = pd.read_csv(r"D:\Passengers flow predictor\Passenger_Flow_Predictor\data\baggage.csv")
staff = pd.read_csv(r"D:\Passengers flow predictor\Passenger_Flow_Predictor\data\staff_shifts.csv")
gate_events = pd.read_csv(r"D:\Passengers flow predictor\Passenger_Flow_Predictor\data\gate_events.csv")
maintenance = pd.read_csv(r"D:\Passengers flow predictor\Passenger_Flow_Predictor\data/maintenance_logs.csv")
retail = pd.read_csv(r"D:\Passengers flow predictor\Passenger_Flow_Predictor\data\retail_transactions.csv")

print(f" Flights loaded: {flights.shape[0]} rows, {flights.shape[1]} columns")
print(f" Passengers: {passengers.shape[0]} rows")
print(f" Security Screening: {security.shape[0]} rows")
print(f" Baggage: {baggage.shape[0]} rows")
print(f" Staff Shifts: {staff.shape[0]} rows")
print(f" Gate Events: {gate_events.shape[0]} rows")
print(f" Maintenance Logs: {maintenance.shape[0]} rows")
print(f" Retail Transactions: {retail.shape[0]} rows")

# Parse datetime columns for flights

print("\n Parsing datetime columns...")

flights['scheduled_departure'] = pd.to_datetime(flights['scheduled_departure'])
flights['scheduled_arrival'] = pd.to_datetime(flights['scheduled_arrival'])
flights['actual_departure'] = pd.to_datetime(flights['actual_departure'])
flights['actual_arrival'] = pd.to_datetime(flights['actual_arrival'])
flights['gate_assignment_time'] = pd.to_datetime(flights['gate_assignment_time'])

print("DateTime parsing complete!")

# Calculate actual delay from timestamps 

print("\n Calculating actual delay metrics...")

# Calculate true delay in minutes from timestamp difference
flights['actual_delay_minutes'] = (
    (flights['actual_departure'] - flights['scheduled_departure']).dt.total_seconds() / 60
).astype(int)

# Note: Original 'delay_minutes' column appears to be gate/boarding delay
flights.rename(columns={'delay_minutes': 'gate_delay_minutes'}, inplace=True)

print(f" Actual delay calculated")
print(f"  - Min delay: {flights['actual_delay_minutes'].min()} minutes")
print(f"  - Max delay: {flights['actual_delay_minutes'].max()} minutes")
print(f"  - Mean delay: {flights['actual_delay_minutes'].mean():.2f} minutes")


# Add derived features for analysis

print("\n Creating derived features...")

# Flight duration (scheduled vs actual)
flights['scheduled_duration_minutes'] = (
    (flights['scheduled_arrival'] - flights['scheduled_departure']).dt.total_seconds() / 60
).astype(int)

flights['actual_duration_minutes'] = (
    (flights['actual_arrival'] - flights['actual_departure']).dt.total_seconds() / 60
).astype(int)

# Classification: On-time or delayed
flights['is_delayed'] = flights['actual_delay_minutes'] > 0

print(f" Derived features added")
print(f"  - {flights['is_delayed'].sum()} delayed flights")
print(f"  - {(~flights['is_delayed']).sum()} on-time flights")

# Summary and Quality Check

print("\n" + "="*80)
print("FLIGHTS DATA - READY FOR ANALYSIS")
print("="*80)

print(f"\n Dataset Shape: {flights.shape}")
print(f"\n Key Columns Available:")
key_cols = ['flight_id', 'airline_name', 'origin', 'destination', 
            'scheduled_departure', 'actual_departure', 'actual_delay_minutes', 
            'gate_delay_minutes', 'delay_reason', 'flight_status', 'season']
for col in key_cols:
    if col in flights.columns:
        print(f"   {col}")

print(f"\n Sample Data - First 5 Flights:")
display(flights[['flight_id', 'airline_name', 'origin', 'destination', 
                  'scheduled_departure', 'actual_departure', 'actual_delay_minutes', 
                  'delay_reason', 'is_delayed']].head())



print(f"\n Data preparation complete! Ready for passenger flow prediction model.")


 Loading datasets...
 Flights loaded: 1000 rows, 32 columns
 Passengers: 2500 rows
 Security Screening: 2500 rows
 Baggage: 2800 rows
 Staff Shifts: 600 rows
 Gate Events: 1200 rows
 Maintenance Logs: 400 rows
 Retail Transactions: 3000 rows

 Parsing datetime columns...
DateTime parsing complete!

 Calculating actual delay metrics...
 Actual delay calculated
  - Min delay: 120 minutes
  - Max delay: 600 minutes
  - Mean delay: 357.24 minutes

 Creating derived features...
 Derived features added
  - 1000 delayed flights
  - 0 on-time flights

FLIGHTS DATA - READY FOR ANALYSIS

 Dataset Shape: (1000, 36)

 Key Columns Available:
   flight_id
   airline_name
   origin
   destination
   scheduled_departure
   actual_departure
   actual_delay_minutes
   gate_delay_minutes
   delay_reason
   flight_status
   season

 Sample Data - First 5 Flights:


,flight_id,airline_name,origin,destination,scheduled_departure,actual_departure,actual_delay_minutes,delay_reason,is_delayed
0,UK-633,Vistara,DEL,SIN,2024-11-11 15:23:02,2024-11-11 17:23:02,120,ATC,True
1,BA-6017,British Airways,DEL,DXB,2024-10-05 07:49:26,2024-10-05 14:49:26,420,CREW,True
2,BA-7303,British Airways,DEL,DXB,2024-11-29 13:09:18,2024-11-29 15:09:18,120,TECH,True
3,KL-7243,KLM,DEL,MAA,2024-11-15 15:07:07,2024-11-15 22:07:07,420,WX,True
4,SG-1280,SpiceJet,DEL,KUL,2024-11-21 19:06:25,2024-11-21 23:06:25,240,WX,True



 Data preparation complete! Ready for passenger flow prediction model.
